In [1]:
import sys
from pathlib import Path

# Добавляем корень проекта в sys.path, если нужно
sys.path.insert(0, str(Path.cwd()))

import os
import yaml
from typing import List, Optional
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

# Загружаем переменные из .env (там должен быть OPENAI_API_KEY)
load_dotenv()

# Проверяем, что ключ есть (если используешь OpenAI)
# print(os.getenv("OPENAI_API_KEY"))

True

In [2]:
from typing import TypedDict, List, Optional

class NarratorState(TypedDict):
    event_type: str
    world: str
    players_alive: List[str]
    event_data: dict
    history: str
    daily_fact: Optional[str]
    messages: Optional[List]   # SystemMessage, HumanMessage
    response: Optional[str]

In [3]:
def load_prompts(world: str) -> dict:
    """Загружает YAML-файл с промптами для указанного мира."""
    base = Path("narrator/prompts")
    if world == "cyberpunk":
        path = base / "cyberpunk.yaml"
    elif world == "medieval":
        path = base / "medieval.yaml"
    else:
        raise ValueError(f"Unknown world: {world}")
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

In [4]:
# Тест загрузки
cyber = load_prompts("cyberpunk")
print(cyber.keys())  # должно быть ['system', 'events']
print(cyber['events'].keys())

dict_keys(['system', 'events'])
dict_keys(['day_start', 'voting', 'player_lynched', 'night_start', 'night_kill', 'sheriff_check', 'doctor_save', 'game_over'])


In [5]:
def select_prompt(state: NarratorState) -> NarratorState:
    prompts = load_prompts(state["world"])
    system_text = prompts["system"]
    
    # Безопасно получаем шаблон события, если нет - fallback
    event_template = prompts["events"].get(
        state["event_type"],
        "Расскажи о событии: {event_type}."
    )
    
    # Форматируем шаблон, подставляя переменные из состояния
    user_text = event_template.format(
        event_type=state["event_type"],
        players_alive=", ".join(state["players_alive"]),
        **state["event_data"],
        history=state["history"],
        daily_fact=state.get("daily_fact", "")
    )
    
    state["messages"] = [
        SystemMessage(content=system_text),
        HumanMessage(content=user_text)
    ]
    return state

In [6]:
def call_model(state: NarratorState) -> NarratorState:
    # Заглушка (для теста без API)
    # state["response"] = f"[Mock LLM] Событие: {state['event_type']}, мир: {state['world']}"
    # return state
    
    # Реальный вызов (если настроен ключ)
    from langchain_groq import ChatGroq
    GROQ_API_KEY = os.environ["GROQ_API_KEY"]
    llm = ChatGroq(
        model=os.getenv("LLM_MODEL", "llama-3.1-8b-instant"),
        temperature=0.8,
        max_tokens=300,
        api_key=GROQ_API_KEY
    )
    response = llm.invoke(state["messages"])
    state["response"] = response.content
    return state

In [7]:
def validate(state: NarratorState) -> NarratorState:
    forbidden = ["мафия — это", "доктор —", "шериф", "роль"]
    for word in forbidden:
        if word in state["response"].lower():
            state["response"] = "Ведущий хранит молчание..."
            break
    return state

In [8]:
def build_graph():
    builder = StateGraph(NarratorState)
    builder.add_node("select_prompt", select_prompt)
    builder.add_node("call_model", call_model)
    builder.add_node("validate", validate)
    
    builder.set_entry_point("select_prompt")
    builder.add_edge("select_prompt", "call_model")
    builder.add_edge("call_model", "validate")
    builder.add_edge("validate", END)
    
    return builder.compile()

graph = build_graph()
print("Граф скомпилирован")

Граф скомпилирован


In [9]:
async def generate(event: dict, world: str, context: dict) -> str:
    state: NarratorState = {
        "event_type": event["type"],
        "world": world,
        "players_alive": context.get("players_alive", []),
        "event_data": event.get("data", {}),
        "history": context.get("history", ""),
        "daily_fact": context.get("daily_fact", ""),
        "messages": None,
        "response": None
    }
    final_state = await graph.ainvoke(state)
    return final_state["response"]

In [10]:
text = await generate(
    event={"type": "night_kill", "data": {"victim": "Джонни"}},
    world="cyberpunk",
    context={
        "players_alive": ["Алиса", "Боб", "Клара"],
        "history": "Вчера казнили Еву.",
        "daily_fact": "Корп «НейроДайн» выпустил новый имплант."
    }
)
print(text)

Гляньте на аутентификацию... Джонни - он не просто "отключился". Кто-то отключил его напрямую из системы. Сейчас Алиса, Боб и Клара - это все, что осталось из группы. Уровень доверия - критический.

Алиса сообщает, что она обнаружила следы дроид-вируса, но не может определить, кто его запустил. Боб утверждает, что он видел странный пакет данных перед тем, как Джонни исчез. Клара просто сидит и повторяет: "Он не мерт, он просто в чат-руме".

Система сигнализирует о предельном уровне активности в сети. Нечто готовится к выпадению из тени.

Вы должны решить, что делать дальше. Давайте сэкономим время? Выберите действия:

1. Проверить лог-файлы системы
2. Выдать задачу Бобу по расследованию
3. Попробовать связаться с Алисой напрямую
4. Сказать Кларе, что она права. Джонни мерт.


In [11]:
events = [
    ("day_start", {}),
    ("voting", {}),
    ("player_lynched", {"victim": "Марк"}),
    ("night_start", {}),
    ("night_kill", {"victim": "Лиза"}),
    ("sheriff_check", {"target": "Олег"}),
    ("doctor_save", {"target": "Анна"}),
    ("game_over", {"winner": "Мирные"})
]

worlds = ["cyberpunk", "medieval"]
results = []

for world in worlds:
    for etype, data in events:
        text = await generate(
            event={"type": etype, "data": data},
            world=world,
            context={"players_alive": ["A", "B", "C"], "history": "", "daily_fact": ""}
        )
        results.append((world, etype, text))
        print(f"--- {world.upper()} | {etype} ---")
        print(text[:200] + "...")
        print()

--- CYBERPUNK | day_start ---
Пожизненный цикл. Звезды просыпаются. Три терминала оживляются. А, Б, С - три брелка в мошне корпорации «Омникульт». 

Прокручивая логи, вижу, что голосование за отключение узла начнется через 30 секу...

--- CYBERPUNK | voting ---
Хакерская подсистема активирована. Анализирую данные сети... 

uzl-A: активность повышена из-за дискуссии по теме "эффективность био-апгрейда". Увеличение шанса на ошибку. 
uzl-B: сигналы анонимности ...

--- CYBERPUNK | player_lynched ---
"Марк деактивирован. Эпизод заканчивается. Но вопрос начинается. Чувствую, что кто-то ждёт в тени. Даты и часы начинают течь: 23:14, 23:15, 23:16... Хакерская хватка в сети усиливается. А, Б, С - три ...

--- CYBERPUNK | night_start ---
Подключение... Подключение... Данные синхронизированы. Система готова к выполнению заданных функций. Обратите внимание: уровень энергопитания снижается. Пороги критических запасов превышены. Следите з...

--- CYBERPUNK | night_kill ---
Система восстановила функ

In [13]:
import pandas as pd
df = pd.DataFrame(results, columns=["world", "event_type", "response"])
df.to_csv("narrator_test_results.csv", index=False)
print("Результаты сохранены в narrator_test_results.csv")

Результаты сохранены в narrator_test_results.csv
